In [1]:
import argparse
import csv
import json
import logging
import random, re, shutil, tempfile, psutil, subprocess
import gzip, os, time, glob, platform
from dataclasses import asdict, dataclass, field
from datetime import datetime, timedelta
from io import BytesIO
from pathlib import Path
from typing import Optional
from urllib.parse import urlparse
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse
import pandas as pd
from difflib import SequenceMatcher
 
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager
import xml.etree.ElementTree as ET
import undetected_chromedriver as uc

# ── logging ────────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("tiket_scraper.log", encoding="utf-8"),
    ],
)
log = logging.getLogger(__name__)

# ── constants ──────────────────────────────────────────────────────────────────
# SITEMAP_INDEX_URL = "https://www.tiket.com/sitemap/id-id/index.xml.gz"
BASE_URL          = "https://www.tiket.com/en-id/hotel/"
LOCALE            = "id"
PAGE_TIMEOUT      = 20
MAX_PROPERTIES    = 25
DRIVER_RECYCLE_EVERY = 5
# Linux → VPS mode (Xvfb); anything else (e.g. local Windows) → headed local mode
IS_VPS = platform.system() == "Linux"
 
# Child sitemap name patterns that contain property detail pages (PDP)
PDP_PATTERNS = {
    "hotel":     ["hotel-pdp"],
    "villa":     ["homes-villa"],
    "homes":     ["homes-pdp"],
    "glamping":  ["homes-glamping"],
    "cottage":   ["homes-cottage"],
    "apartment": ["homes-apartment"],
}

def ensure_virtual_display():
    """
    Start Xvfb virtual display if not already running.
    Sets DISPLAY env var so Chrome picks it up automatically.
    """
    display = ":99"

    # Check if Xvfb is already running on this display
    result = subprocess.run(
        ["pgrep", "-f", f"Xvfb {display}"],
        capture_output=True, text=True
    )

    if result.returncode != 0:
        # Not running — start it
        log.info(f"Starting Xvfb on display {display}...")
        subprocess.Popen(
            ["Xvfb", display, "-screen", "0", "1920x1080x24"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )
         # ── Wait until Xvfb socket actually exists, not just a fixed sleep ──
        socket_path = f"/tmp/.X11-unix/X{display.lstrip(':')}"
        for _ in range(20):          # up to 10s
            if os.path.exists(socket_path):
                log.info(f"Xvfb socket ready: {socket_path}")
                break
            time.sleep(0.5)
        else:
            raise RuntimeError("Xvfb did not start in time — socket never appeared")
    else:
        log.info(f"Xvfb already running on {display}")

    # CRITICAL: set DISPLAY so Chrome finds the virtual screen
    os.environ["DISPLAY"] = display
    log.info(f"DISPLAY set to {display}")

def clear_uc_cache():
    """Delete undetected_chromedriver's cached patched driver.

    UC caches the patched chromedriver (e.g. under %APPDATA%\\undetected_chromedriver
    on Windows). If a wrong-version driver got cached (the v150 vs Chrome-149 error),
    it keeps reusing it. Removing it forces UC to re-download a matching driver.
    """
    try:
        patcher = uc.Patcher()
        exe = getattr(patcher, "executable_path", None)
        if exe and os.path.exists(exe):
            try:
                os.remove(exe)
                log.info(f"Cleared UC cached driver: {exe}")
            except PermissionError:
                log.warning(f"Could not remove {exe} (locked); attempting dir wipe")
        data_dir = getattr(patcher, "data_path", None)
        if data_dir and os.path.isdir(data_dir):
            shutil.rmtree(data_dir, ignore_errors=True)
            log.info(f"Cleared UC data dir: {data_dir}")
    except Exception as e:
        log.warning(f"Could not clear UC cache: {e}")

def get_chrome_version() -> int:
    """Auto-detect installed Chrome major version — no more hardcoding version_main."""
    # ── Windows: registry first, then chrome.exe file metadata ────────────────
    if platform.system() == "Windows":
        try:
            import winreg
            with winreg.OpenKey(winreg.HKEY_CURRENT_USER, r"Software\Google\Chrome\BLBeacon") as key:
                version_str, _ = winreg.QueryValueEx(key, "version")  # e.g. "149.0.7827.201"
                major = int(version_str.split(".")[0])
                log.info(f"Auto-detected Chrome (registry): {version_str} → major version {major}")
                return major
        except Exception as e:
            log.warning(f"Registry Chrome detection failed: {e}")

        # Fallback: read version straight from chrome.exe file metadata
        chrome_paths = [
            os.path.expandvars(r"%ProgramFiles%\Google\Chrome\Application\chrome.exe"),
            os.path.expandvars(r"%ProgramFiles(x86)%\Google\Chrome\Application\chrome.exe"),
            os.path.expandvars(r"%LocalAppData%\Google\Chrome\Application\chrome.exe"),
        ]
        for path in chrome_paths:
            if os.path.exists(path):
                try:
                    result = subprocess.run(
                        ["powershell", "-NoProfile", "-Command",
                         f"(Get-Item '{path}').VersionInfo.ProductVersion"],
                        capture_output=True, text=True, timeout=10
                    )
                    if result.returncode == 0 and result.stdout.strip():
                        version_str = result.stdout.strip()
                        major = int(version_str.split(".")[0])
                        log.info(f"Auto-detected Chrome (exe): {version_str} → major version {major}")
                        return major
                except Exception as e:
                    log.warning(f"Exe Chrome detection failed for {path}: {e}")
        return None

    # ── Linux / VPS: try common Chrome binary names ───────────────────────────
    try:
        for binary in ["google-chrome", "google-chrome-stable", "chromium-browser", "chromium"]:
            result = subprocess.run(
                [binary, "--version"],
                capture_output=True, text=True, timeout=5
            )
            if result.returncode == 0:
                version_str = result.stdout.strip()  # e.g. "Google Chrome 136.0.7103.93"
                major = int(version_str.split()[-1].split(".")[0])
                log.info(f"Auto-detected Chrome: {version_str} → major version {major}")
                return major
    except Exception as e:
        log.warning(f"Could not auto-detect Chrome version: {e}")
    return None   # let UC figure it out itself


def build_driver(download_folder: str, headless: bool):
    # ── On VPS: spin up virtual display so Chrome runs headed ────────────────
    if IS_VPS:
        ensure_virtual_display()

    opts = uc.ChromeOptions()

    prefs = {
        "download.default_directory":         download_folder,
        "download.prompt_for_download":       False,
        "download.directory_upgrade":         True,
        "safebrowsing.enabled":               True,
        "plugins.always_open_pdf_externally": True,
        "intl.accept_languages":              "en-US,en",
    }
    opts.add_experimental_option("prefs", prefs)

    # ── Stability flags (still needed even in headed mode on VPS) ────────────
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--disable-gpu")              # VPS still has no real GPU
    opts.add_argument("--disable-software-rasterizer")
    opts.add_argument("--disable-extensions")
    opts.add_argument("--disable-background-networking")
    opts.add_argument("--no-first-run")
    opts.add_argument("--no-default-browser-check")
    opts.add_argument("--mute-audio")
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument("--start-maximized")
    opts.add_argument("--lang=en-US,en")

    # ── NO --headless flag at all when using Xvfb ────────────────────────────
    # headless=False both here and in uc.Chrome() call

    # ── Temp profile in shared memory ────────────────────────────────────────
    # shm_available = os.path.exists("/dev/shm") and os.access("/dev/shm", os.W_OK)
    # tmp_base = "/dev/shm" if shm_available else "/tmp"
    # tmp_profile = tempfile.mkdtemp(prefix="chrome_profile_", dir=tmp_base)
    # opts.add_argument(f"--user-data-dir={tmp_profile}")
    # log.info(f"Chrome profile dir: {tmp_profile}")

    chrome_version = get_chrome_version()

    def _start_uc():
        driver_kwargs = {
            "options":  opts,
            "headless": False,   # True headed mode — Xvfb provides the display
        }
        if chrome_version:
            driver_kwargs["version_main"] = chrome_version
        return uc.Chrome(**driver_kwargs)

    try:
        try:
            driver = _start_uc()
            log.info("UC Chrome started in headed mode via Xvfb")
        except Exception as first_err:
            # Most common cause: a stale wrong-version chromedriver cached by UC
            # (the "ChromeDriver only supports Chrome version 150" error). Wipe the
            # cache and retry ONCE with the detected version before giving up on UC.
            log.warning(f"UC Chrome first attempt failed: {first_err}")
            log.info("Clearing UC cache and retrying UC once...")
            clear_uc_cache()
            time.sleep(2)
            driver = _start_uc()
            log.info("UC Chrome started on retry (after cache clear)")

    except Exception as e:
        log.error(f"UC Chrome failed: {e}")
        log.info("Falling back to standard Selenium...")
        # shutil.rmtree(tmp_profile, ignore_errors=True)

        from selenium.webdriver.chrome.service import Service
        from webdriver_manager.chrome import ChromeDriverManager

        std_opts = Options()
        std_opts.add_argument("--no-sandbox")
        std_opts.add_argument("--disable-dev-shm-usage")
        std_opts.add_argument("--disable-gpu")
        std_opts.add_argument("--window-size=1920,1080")
        if IS_VPS and not headless:
            pass   # Xvfb is already running, no --headless needed
        elif headless:
            std_opts.add_argument("--headless=new")
        std_opts.add_experimental_option("prefs", prefs)
        std_opts.add_argument(
            "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/149.0.0.0 Safari/537.36"
        )

        driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()),
            options=std_opts
        )
        log.info("Standard Selenium started")

    # driver._tmp_profile = tmp_profile
    return driver


In [8]:
PROJECT_FOLDER   = os.path.join(os.getcwd(), "sitemaps")
CHILD_FOLDER     = os.path.join(os.getcwd(), "child-sitemaps")
OTHER_CHILD_FOLDER = os.path.join(os.getcwd(), "other-child-sitemaps")
CURR_DIR         = os.path.dirname(os.getcwd())
SITEMAP_INDEX    = "https://www.tiket.com/en-id/hotel/"
FOUND_URL_PATH = "found_url.json"

def clean_property_url(url: str) -> str:
    """
    Strip query params and fragment, keep the trailing numeric property ID.
    e.g. https://www.tiket.com/en-id/hotel/indonesia/djineng-rice-terrace-canggu-511001669192830200?room=1&...
    ->   https://www.tiket.com/en-id/hotel/indonesia/djineng-rice-terrace-canggu-511001669192830200
    """
    base = urlunparse(urlparse(url)._replace(query="", fragment=""))
    return base.rstrip("/")

def ask_customer_id() -> int:
    while True:
        raw = input("customer_id (integer): ").strip()
        try:
            return int(raw)
        except ValueError:
            print("  -> not an integer, try again.")

def update_found_url(customer_id: int, cleaned_url: str) -> bool:
    with open(FOUND_URL_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)

    for entry in data:
        if entry.get("customer_id") == customer_id:
            old = entry.get("url", "")
            if old == cleaned_url:
                print(f"[=] customer_id {customer_id} already has this URL, nothing to do.")
                return False

            print(f"    name : {entry.get('properties_name')}")
            print(f"    old  : {old}")
            print(f"    new  : {cleaned_url}")
            if input("    write this change? [y/N]: ").strip().lower() != "y":
                print("[-] skipped.")
                return False

            entry["url"] = cleaned_url
            with open(FOUND_URL_PATH, "w", encoding="utf-8") as f:
                json.dump(data, f, indent=4, ensure_ascii=False)
            print("[+] saved.")
            return True

    print(f"[!] customer_id {customer_id} not found in found_url.json — nothing written.")
    return False


driver = build_driver(CURR_DIR, headless=IS_VPS)
wait = WebDriverWait(driver, PAGE_TIMEOUT)
driver.get("https://www.tiket.com/en-id/hotel/")

try:
    while True:
        print("\n--- Search the property in the browser, then come back here. ---")
        input("Press Enter once the property page is open... ")

        # Switch to the most recently opened tab if multiple tabs exist
        handles = driver.window_handles
        if len(handles) > 1:
            driver.switch_to.window(handles[-1])

        current_url = driver.current_url
        cleaned = clean_property_url(current_url)
        print(f"[url] raw     : {current_url}")
        print(f"[url] cleaned : {cleaned}")

        customer_id = ask_customer_id()
        update_found_url(customer_id, cleaned)

        if input("\nAnother property? [y/N]: ").strip().lower() != "y":
            break
finally:
    driver.quit()



16:19:17 [INFO] Auto-detected Chrome (registry): 151.0.7922.110 → major version 151
16:19:21 [INFO] patching driver executable C:\Users\anton\appdata\roaming\undetected_chromedriver\undetected_chromedriver.exe
16:19:21 [INFO] UC Chrome started in headed mode via Xvfb



--- Search the property in the browser, then come back here. ---
[url] raw     : https://www.tiket.com/en-id/hotel/indonesia/sisilia-lodge-810001761276246892
[url] cleaned : https://www.tiket.com/en-id/hotel/indonesia/sisilia-lodge-810001761276246892
    name : Sisilia Lodge
    old  : 
    new  : https://www.tiket.com/en-id/hotel/indonesia/sisilia-lodge-810001761276246892
[+] saved.

--- Search the property in the browser, then come back here. ---
[url] raw     : https://www.tiket.com/id-id/hotel/indonesia/smart-inn-at-aeropolis-residence-806001750548717693
[url] cleaned : https://www.tiket.com/id-id/hotel/indonesia/smart-inn-at-aeropolis-residence-806001750548717693
    name : Smart Inn
    old  : 
    new  : https://www.tiket.com/id-id/hotel/indonesia/smart-inn-at-aeropolis-residence-806001750548717693
[+] saved.

--- Search the property in the browser, then come back here. ---
[url] raw     : https://www.tiket.com/en-id/homes/indonesia/susi-villas-1-710001727736481142
[url] cleane

KeyboardInterrupt: Interrupted by user